# Sales analysis - a RAD IDE notebook
(c) 2026 Musoftware - https://musoftwares.com - MIT License

This notebook reads `sales.csv` (next to it) and answers a few questions with the **Python standard library only**
(no pandas needed): totals per region and product, the best month, a month-over-month trend and a text bar chart.

*تحليل المبيعات - دفتر Jupyter في RAD IDE: يقرأ ملف sales.csv ويحسب الإجماليات حسب المنطقة والمنتج وأفضل شهر واتجاه المبيعات، باستخدام مكتبة بايثون القياسية فقط.*

Run the cells from top to bottom (Shift+Enter) or **Run All**.

In [ ]:
# (c) 2026 Musoftware - https://musoftwares.com - MIT License
import csv
import statistics
from collections import defaultdict
from pathlib import Path

rows = []
with open(Path("sales.csv"), encoding="utf-8") as f:
    for r in csv.DictReader(f):
        r["Units"] = int(r["Units"])
        r["Price"] = float(r["Price"])
        r["Revenue"] = r["Units"] * r["Price"]
        rows.append(r)
print(len(rows), "rows,", len({r['Month'] for r in rows}), "months")

## Revenue per region and per product

In [ ]:
def total_by(key):
    out = defaultdict(float)
    for r in rows:
        out[r[key]] += r["Revenue"]
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

by_region = total_by("Region")
by_product = total_by("Product")
for name, value in by_region.items():
    print(f"{name:<12}{value:>10,.2f}")
print()
for name, value in by_product.items():
    print(f"{name:<12}{value:>10,.2f}")

## Best month and month-over-month change

In [ ]:
by_month = dict(sorted(total_by("Month").items()))
best = max(by_month, key=by_month.get)
print("best month:", best, f"{by_month[best]:,.2f}")
months = list(by_month)
for prev, cur in zip(months, months[1:]):
    change = (by_month[cur] - by_month[prev]) / by_month[prev] * 100
    print(f"{cur}: {by_month[cur]:>9,.2f}  ({change:+.1f} %)")
print("average month:", f"{statistics.mean(by_month.values()):,.2f}",
      " stdev:", f"{statistics.stdev(by_month.values()):,.2f}")

## A text bar chart
Every `#` is about 2 % of the best month.

In [ ]:
top = max(by_month.values())
for month, value in by_month.items():
    print(f"{month}  {'#' * round(value / top * 50):<50}  {value:,.0f}")

## Your turn
Change `key` to `"Product"` or `"Region"` and run the cell again.

In [ ]:
key = "Region"
summary = total_by(key)
share = {k: v / sum(summary.values()) * 100 for k, v in summary.items()}
share